# Backward-pass capture with TorchLens

Backward capture records the autograd graph that PyTorch executes after a logged forward pass. It lets you inspect per-layer output grads, traverse `grad_fn` topology, connect backward nodes to forward layers, validate captured grads, render the backward graph, and stream grad tensors to disk when RAM matters.


In [ ]:
from pathlib import Path
import sys
import tempfile

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import torch
from torch import nn
import torchlens as tl


torch.manual_seed(0)


class TinyBackwardNet(nn.Module):
    """Small network used throughout this tutorial."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.fc1 = nn.Linear(4, 5)
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the network."""

        hidden = torch.relu(self.fc1(x))
        viewed = hidden.view(hidden.shape[0], 5)
        return self.fc2(viewed)


def fresh_model_and_input() -> tuple[TinyBackwardNet, torch.Tensor]:
    """Return a deterministic model/input pair."""

    torch.manual_seed(7)
    return TinyBackwardNet(), torch.randn(3, 4, has_trainable_params=True)


def logged_with_backward(**kwargs: object) -> tl.Trace:
    """Capture a forward pass and then a scalar backward pass."""

    model, x = fresh_model_and_input()
    trace = tl.trace(model, x, layers_to_save="all", save_grads=True, **kwargs)
    loss = trace[trace.output_layers[0]].out.sum()
    trace.log_backward(loss)
    return trace

## Pattern 1: basic grad access per layer

Call `trace(..., save_grads=True)`, build a scalar loss from a saved out, and pass it to `trace.log_backward(loss)`. Saved layer grads are then available through normal layer lookup.


In [ ]:
trace = logged_with_backward()

assert trace.has_grads
assert trace.ops_with_saved_grads
first_grad_layer = trace.ops_with_saved_grads[0]
first_grad = trace[first_grad_layer].grad

assert isinstance(first_grad, torch.Tensor)
assert first_grad.shape == trace[first_grad_layer].shape
first_grad_layer, tuple(first_grad.shape)

## Pattern 2: backward graph topology

The `grad_fns` accessor exposes the backward graph in discovery order. Intervening nodes are autograd operations that do not have a one-to-one forward layer match.


In [ ]:
grad_fn_count = len(trace.grad_fns)
intervening = [grad_fn for grad_fn in trace.grad_fns if grad_fn.has_op]
root = trace.grad_fn_logs[trace.backward_root_grad_fn_id]

assert grad_fn_count > 0
assert intervening
assert root.next_grad_fn_ids

grad_fn_count, root.label, len(intervening)

## Pattern 3: cross-references between forward and backward

Forward layers and backward grad-fn logs point back to each other when TorchLens can identify the correspondence.


In [ ]:
paired_grad_fns = [grad_fn for grad_fn in trace.grad_fns if grad_fn.op is not None]

assert paired_grad_fns
for grad_fn in paired_grad_fns:
    assert grad_fn.op.grad_fn_log is grad_fn

paired_grad_fns[0].op.layer_label, paired_grad_fns[0].label

## Pattern 4: per-layer grad selection

`grads_to_save` can be independent from `layers_to_save`. When `grads_to_save` is omitted but backward capture is enabled, TorchLens defaults it from `layers_to_save`.


In [ ]:
model, x = fresh_model_and_input()
selected_log = tl.trace(
    model,
    x,
    layers_to_save=["relu"],
    grads_to_save=["relu"],
)
selected_log.log_backward(selected_log[selected_log.output_layers[0]].out.sum())

assert selected_log.ops_with_saved_grads
assert all("relu" in label for label in selected_log.ops_with_saved_grads)

model, x = fresh_model_and_input()
defaulted_log = tl.trace(model, x, layers_to_save=["relu"], save_grads=True)
defaulted_log.log_backward(defaulted_log[defaulted_log.output_layers[0]].out.sum())

assert defaulted_log.ops_with_saved_grads == selected_log.ops_with_saved_grads
selected_log.ops_with_saved_grads

## Pattern 5: multi-loss capture with `recording_backward()`

Use `recording_backward()` when you want to manage `.backward()` calls yourself, including `retain_graph=True` for multiple losses on the same graph.


In [ ]:
model, x = fresh_model_and_input()
multi_log = tl.trace(model, x, layers_to_save="all", save_grads=True)
output = multi_log[multi_log.output_layers[0]].out
loss_a = output[:, 0].sum()
loss_b = output[:, 1].sum()

with multi_log.recording_backward():
    loss_a.backward(retain_graph=True)
    loss_b.backward()

assert multi_log.backward_num_calls == 2
assert all(grad_fn.num_calls >= 1 for grad_fn in multi_log.grad_fns)
multi_log.backward_num_calls

## Pattern 6: backward visualization

`show_backward_graph()` renders the captured autograd graph. `save_only=True` is useful in notebooks and CI because it writes a file without opening a viewer.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    out_base = Path(tmpdir) / "backward_graph"
    dot_source = tl.draw_backward(
        trace,
        vis_outpath=str(out_base),
        vis_save_only=True,
        vis_fileformat="pdf",
    )
    rendered = out_base.with_suffix(".pdf")
    assert rendered.exists()
    assert rendered.stat().st_size > 0
    assert "digraph" in dot_source

## Pattern 7: `validate_backward_pass`

Validation compares TorchLens backward capture against a stock autograd run. The perturbation option intentionally corrupts saved grads and should fail.


In [ ]:
model, x = fresh_model_and_input()
assert tl.validate_backward_pass(model, x)

model, x = fresh_model_and_input()
assert not tl.validate_backward_pass(model, x, perturb_saved_grads=True)

## Pattern 8: saving and loading a bundle with backward data

Portable bundles include backward flat fields, grad-fn logs, cross-references, and saved grads.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    bundle_path = Path(tmpdir) / "backward_bundle.tl"
    tl.save(trace, bundle_path)
    restored = tl.load(bundle_path, lazy=True)
    label = restored.ops_with_saved_grads[0]

    assert restored.has_backward_pass
    assert restored.backward_num_calls == trace.backward_num_calls
    assert len(restored.grad_fn_logs) == len(trace.grad_fn_logs)
    assert restored.grad_fns[0].ops
    assert isinstance(restored[label].grad, torch.Tensor)

## Pattern 9: streaming grads to disk

`save_grads_to` streams saved grads into a portable bundle and `keep_grads_in_memory=False` leaves lazy refs behind. Accessing `layer.grad` materializes the tensor from disk.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    bundle_path = Path(tmpdir) / "streamed_backward.tl"
    model, x = fresh_model_and_input()
    streamed_log = tl.trace(
        model,
        x,
        layers_to_save="all",
        save_grads=True,
        save_grads_to=bundle_path,
        keep_grads_in_memory=False,
    )
    streamed_log.log_backward(streamed_log[streamed_log.output_layers[0]].out.sum())

    label = streamed_log.ops_with_saved_grads[0]
    assert streamed_log[label].__dict__["grad"] is None
    assert streamed_log[label].grad_ref is not None
    assert isinstance(streamed_log[label].grad, torch.Tensor)

    lazy_streamed = tl.load(bundle_path, lazy=True)
    assert lazy_streamed[label].__dict__["grad"] is None
    assert isinstance(lazy_streamed[label].grad, torch.Tensor)

## Pattern 10: custom `autograd.Function`

Custom autograd functions are preserved in the backward graph and marked with `is_custom`.


In [ ]:
class DoubleFn(torch.autograd.Function):
    """Custom function for tutorial coverage."""

    @staticmethod
    def forward(ctx: torch.autograd.function.FunctionCtx, x: torch.Tensor) -> torch.Tensor:
        """Return doubled input."""

        return x * 2

    @staticmethod
    def backward(ctx: torch.autograd.function.FunctionCtx, grad: torch.Tensor) -> torch.Tensor:
        """Return doubled upstream grad."""

        return grad * 2


class CustomFnNet(nn.Module):
    """Tiny model that uses a custom autograd function."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model."""

        return DoubleFn.apply(x).sum()


custom_log = tl.trace(CustomFnNet(), torch.randn(2, 4, has_trainable_params=True), save_grads=True)
custom_log.log_backward(custom_log[custom_log.output_layers[0]].out)

assert any(grad_fn.is_custom for grad_fn in custom_log.grad_fns)
[grad_fn.label for grad_fn in custom_log.grad_fns if grad_fn.is_custom]

## Gotchas

Higher-order grads: basic `create_graph=True` capture works, but exhaustive higher-order coverage is still deferred.

Inference mode: backward capture needs autograd history. If a workflow opts into `train_mode=True`, active `torch.inference_mode()` is rejected because inference tensors cannot retain that history.

Distributed training: backward data is rank-local. With DDP or multi-rank jobs, capture each rank intentionally and manage synchronization outside TorchLens.

Deferred features: fastlog grads, per-grad-fn memory-cost attribution, and side-by-side forward/backward visualization remain parking-lot items in `todos.md`.


In [ ]:
model, x = fresh_model_and_input()
higher_order_log = tl.trace(model, x, save_grads=True)
higher_order_log.log_backward(
    higher_order_log[higher_order_log.output_layers[0]].out.sum(), create_graph=True
)
assert higher_order_log.backward_num_calls == 1

try:
    with torch.inference_mode():
        model, x = fresh_model_and_input()
        tl.trace(model, x, grads_to_save="all")
except ValueError as exc:
    assert "inference" in str(exc).lower()
else:
    raise AssertionError("inference_mode should reject train-mode backward capture")